1. Data Preprocessing

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score

# Load dataset
anime = pd.read_csv("/content/anime.csv")

anime['rating'] = anime['rating'].astype(str).replace('nan', '', regex=False)

# Now, apply general fillna for other columns. 'rating' is already handled and is a string type.
anime.fillna('', inplace=True)

# Combine features (genres + rating + type)
# The columns 'genre', 'type', and 'rating' are now all string type with NaNs handled,
# so .astype(str) calls are redundant here but harmless.
anime['combined_features'] = (
    anime['genre'] + ' ' +
    anime['type'] + ' ' +
    anime['rating']
)
print(anime)

       anime_id                                               name  \
0         32281                                     Kimi no Na wa.   
1          5114                   Fullmetal Alchemist: Brotherhood   
2         28977                                           Gintama°   
3          9253                                        Steins;Gate   
4          9969                                      Gintama&#039;   
...         ...                                                ...   
12289      9316       Toushindai My Lover: Minami tai Mecha-Minami   
12290      5543                                        Under World   
12291      5621                     Violence Gekiga David no Hoshi   
12292      6133  Violence Gekiga Shin David no Hoshi: Inma Dens...   
12293     26081                   Yasuji no Pornorama: Yacchimae!!   

                                                   genre   type episodes  \
0                   Drama, Romance, School, Supernatural  Movie        1   
1      

2. Feature Extraction

In [2]:
# Convert categorical features into numerical representation
vectorizer = TfidfVectorizer(stop_words='english')
feature_matrix = vectorizer.fit_transform(anime['combined_features'])

# Compute cosine similarity
cosine_sim = cosine_similarity(feature_matrix, feature_matrix)
print(cosine_sim)


[[1.         0.08524165 0.         ... 0.         0.         0.11121693]
 [0.08524165 1.         0.16653466 ... 0.         0.         0.        ]
 [0.         0.16653466 1.         ... 0.         0.         0.        ]
 ...
 [0.         0.         0.         ... 1.         0.3272428  0.23089043]
 [0.         0.         0.         ... 0.3272428  1.         0.21898861]
 [0.11121693 0.         0.         ... 0.23089043 0.21898861 1.        ]]


3. Recommendation Function

In [3]:
def recommend_anime(title, cosine_sim=cosine_sim, threshold=0.0):
    if title not in anime['name'].values:
        return []
    idx = anime[anime['name'] == title].index[0]
    sim_scores = list(enumerate(cosine_sim[idx]))
    # Filter based on threshold and take top N, excluding itself
    filtered_sim_scores = [score for score in sim_scores if score[0] != idx and score[1] > threshold]
    sim_scores = sorted(filtered_sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[0:10]  # Top 10 recommendations after filtering
    anime_indices = [i[0] for i in sim_scores]
    return anime['name'].iloc[anime_indices].tolist()

# Example usage
print("Recommendations for Naruto (threshold=0.0):")
print(recommend_anime("Naruto", threshold=0.0))
print("Recommendations for Naruto (threshold=0.2):")
print(recommend_anime("Naruto", threshold=0.2))

Recommendations for Naruto (threshold=0.0):
['Iron Virgin Jun', 'Naruto: Shippuuden Movie 3 - Hi no Ishi wo Tsugu Mono', 'Dragon Ball Super', 'Ikkitousen: Extravaganza Epoch', 'Tenjou Tenge', 'Naruto: Shippuuden', 'Gakuen Tokusou Hikaruon', 'Rekka no Honoo', 'Naruto x UT', 'Dragon Ball Z']
Recommendations for Naruto (threshold=0.2):
['Iron Virgin Jun', 'Naruto: Shippuuden Movie 3 - Hi no Ishi wo Tsugu Mono', 'Dragon Ball Super', 'Ikkitousen: Extravaganza Epoch', 'Tenjou Tenge', 'Naruto: Shippuuden', 'Gakuen Tokusou Hikaruon', 'Rekka no Honoo', 'Naruto x UT', 'Dragon Ball Z']


4. Evaluation

In [8]:
from sklearn.model_selection import train_test_split
import numpy as np

# Split dataset
train, test = train_test_split(anime, test_size=0.2, random_state=42)

all_recommendation_similarities = []
num_evaluated_items = 0

# Initialize for precision, recall, f1 calculation
total_tp = 0
total_fp = 0
total_fn = 0

relevance_threshold_for_ground_truth = 0.7 # Similarity score above which an item is considered relevant in ground truth
recommendation_threshold = 0.1 # Threshold used by recommend_anime for generating recommendations

# Evaluate a sample from the test set
# Iterate through test set items and get recommendations
for i, row in test.head(50).iterrows(): # Take a subset for quicker evaluation
    anime_title = row['name']
    original_idx = anime[anime['name'] == anime_title].index[0]

    # Get recommendations for the current anime
    recommendations_list = recommend_anime(anime_title, cosine_sim=cosine_sim, threshold=recommendation_threshold)

    if recommendations_list:
        # Get the indices of the recommended animes
        recommended_indices = set(anime[anime['name'].isin(recommendations_list)].index)

        # Calculate the similarity scores between the original anime and its recommendations
        current_similarities = cosine_sim[original_idx, list(recommended_indices)]
        all_recommendation_similarities.extend(current_similarities)

        # Define ground truth relevant items based on a higher similarity threshold
        # Get all items (excluding itself) that are above the relevance_threshold_for_ground_truth
        ground_truth_relevant_indices_array = np.where(cosine_sim[original_idx] > relevance_threshold_for_ground_truth)[0]
        # Filter out the original item itself if it's included in the highly similar items
        ground_truth_relevant_indices = set([idx for idx in ground_truth_relevant_indices_array if idx != original_idx])

        # Calculate TP, FP, FN for this item
        tp = len(recommended_indices.intersection(ground_truth_relevant_indices))
        fp = len(recommended_indices - ground_truth_relevant_indices)
        fn = len(ground_truth_relevant_indices - recommended_indices)

        # Accumulate for overall metrics
        total_tp += tp
        total_fp += fp
        total_fn += fn
        num_evaluated_items += 1

if num_evaluated_items > 0:
    average_recommendation_similarity = np.mean(all_recommendation_similarities)
    print(f"Average similarity of recommended items (recommendation threshold={recommendation_threshold}): {average_recommendation_similarity:.4f}")

    # Calculate overall precision, recall, F1
    precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
    recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    print(f"Precision (ground truth relevance threshold={relevance_threshold_for_ground_truth}): {precision:.4f}")
    print(f"Recall (ground truth relevance threshold={relevance_threshold_for_ground_truth}): {recall:.4f}")
    print(f"F1-Score (ground truth relevance threshold={relevance_threshold_for_ground_truth}): {f1:.4f}")
else:
    print("No recommendations were generated for evaluation.")

Average similarity of recommended items (recommendation threshold=0.1): 0.7879
Precision (ground truth relevance threshold=0.7): 0.7678
Recall (ground truth relevance threshold=0.7): 0.4499
F1-Score (ground truth relevance threshold=0.7): 0.5673


5. Interview Questions

In [5]:
print("Q1: User-based filtering recommends items based on similarity between users, while item-based filtering recommends items based on similarity between items.")
print("Q2: Collaborative filtering is a method that makes automatic predictions about a user's interests by collecting preferences from many users.")

Q1: User-based filtering recommends items based on similarity between users, while item-based filtering recommends items based on similarity between items.
Q2: Collaborative filtering is a method that makes automatic predictions about a user's interests by collecting preferences from many users.
